# Wallet Strategy Selection

Unified selection stage: runs **all wallet-selection methods** and saves each
as a named `WalletSelectionStrategy` artifact in the workspace.  The backtest
stage loads these artifacts directly — no strategy logic is re-built there.

## Methods included
| id suffix | description |
|-----------|-------------|
| `skill_sweep` → `quality_core` | top-N by best skill metric (grid-searched on train-a→train-b) |
| `cohort_early_entry` | wallets that enter markets early |
| `cohort_smooth_pnl` | wallets with high PnL relative to volatility |
| `volatility` | volatility-filtered wallets (from the profitable_wallet_analysis path) |

Each method produces **three trigger variants**:
- `score_threshold`  — composite signal score ≥ calibrated threshold (Kelly sizing)
- `all_open_buys`    — every open-buy event (fixed stake baseline)
- `copy_triggers`    — every trade (open/add/close/reduce), tight slippage


In [1]:
%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.2f}'.format

## Configuration

In [2]:
# ── configuration ─────────────────────────────────────────────────────────────
RECENCY_DAYS     = 90

PRICE_BUCKET_BINS   = [0.0, 0.1, 0.25, 0.4, 0.6, 0.75, 0.9, 1.0]
PRICE_BUCKET_LABELS = [
    "0.00-0.10", "0.10-0.25", "0.25-0.40", "0.40-0.60",
    "0.60-0.75", "0.75-0.90", "0.90-1.00",
]

TRADES_DIR    = Path("../../data/polygon_trades_processed")
WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

dataset = ds.dataset(TRADES_DIR, format="parquet")
DATASET_COLUMNS        = set(dataset.schema.names)
TRIGGER_TX_HASH_COLUMN = (
    "transaction_hash" if "transaction_hash" in DATASET_COLUMNS
    else ("tx_hash" if "tx_hash" in DATASET_COLUMNS else None)
)

METRICS_A_PATH          = WORKSPACE_DIR / "wallet_metrics_train_a.parquet"
METRICS_B_PATH          = WORKSPACE_DIR / "wallet_metrics_train_b.parquet"
METRICS_FULL_PATH       = WORKSPACE_DIR / "wallet_metrics_full_train.parquet"
METRICS_TEST_PATH       = WORKSPACE_DIR / "wallet_metrics_test.parquet"
CALIBRATION_SIGNALS_PATH = WORKSPACE_DIR / "signal_events_train_b.parquet"
TEST_SIGNALS_PATH       = WORKSPACE_DIR / "signal_events_test.parquet"


## Derive train/test dates from data

In [3]:
# ── derive train/test boundary from is_train flag ───────────────────────────
_sample = pd.read_parquet(sorted(TRADES_DIR.glob("*.parquet"))[0], columns=["dt", "is_train"])
_sample["dt"] = pd.to_datetime(_sample["dt"], utc=True)
END_DATE_TRAIN   = _sample.loc[_sample["is_train"], "dt"].max().date()
TRAIN_START_DATE = _sample.loc[_sample["is_train"], "dt"].min().date()
# Split train_a / train_b at the trade-count median of training rows so that
# both halves always contain roughly equal numbers of trades even when the
# data is unevenly distributed over calendar time.
_train_rows = _sample.loc[_sample["is_train"]].sort_values("dt")
TRAIN_A_END_DATE = _train_rows["dt"].quantile(0.5, interpolation="nearest").date()
del _sample, _train_rows
print(f"END_DATE_TRAIN={END_DATE_TRAIN}  TRAIN_A_END_DATE={TRAIN_A_END_DATE}")


END_DATE_TRAIN=2026-02-15  TRAIN_A_END_DATE=2026-01-26


## Compute / load wallet skill metrics

In [4]:
from polymarket_analysis.wallet_selection.metrics import compute_wallet_skill_workspace

if all(p.exists() for p in [METRICS_A_PATH, METRICS_B_PATH, METRICS_FULL_PATH, METRICS_TEST_PATH]):
    train_a_metrics    = pd.read_parquet(METRICS_A_PATH)
    train_b_metrics    = pd.read_parquet(METRICS_B_PATH)
    full_train_metrics = pd.read_parquet(METRICS_FULL_PATH)
    test_metrics       = pd.read_parquet(METRICS_TEST_PATH)
else:
    train_a_metrics, train_b_metrics, full_train_metrics, test_metrics = (
        compute_wallet_skill_workspace(
            dataset,
            end_date_train=END_DATE_TRAIN,
            train_a_end_date=TRAIN_A_END_DATE,
            recency_days=RECENCY_DAYS,
        )
    )
    train_a_metrics.to_parquet(METRICS_A_PATH, index=False)
    train_b_metrics.to_parquet(METRICS_B_PATH, index=False)
    full_train_metrics.to_parquet(METRICS_FULL_PATH, index=False)
    test_metrics.to_parquet(METRICS_TEST_PATH, index=False)

pd.Series({
    "train_a_wallets":    len(train_a_metrics),
    "train_b_wallets":    len(train_b_metrics),
    "full_train_wallets": len(full_train_metrics),
    "test_wallets":       len(test_metrics),
}).to_frame("value")


,value
train_a_wallets,10964
train_b_wallets,10964
full_train_wallets,10964
test_wallets,10964


## Cohort selection sweep (skill path)

Grid-search over metrics × top-N using train-a → train-b persistence.

In [5]:
from polymarket_analysis.wallet_selection.selector import (
    CANDIDATE_METRICS,
    cohort_selection_sweep,
    select_wallets,
)

cohort_sweep = cohort_selection_sweep(train_a_metrics, train_b_metrics, CANDIDATE_METRICS)
cohort_sweep.sort_values(
    ["train_b_avg_copy_roi_capped", "train_b_weighted_prob_edge"], ascending=False
).head(20)


,metric,top_n,wallets_found_in_train_b,train_b_open_buy_trades,train_b_weighted_prob_edge,train_b_avg_prob_edge,train_b_avg_copy_roi_capped,train_b_total_copy_pnl_usdc,train_b_hit_rate
10,avg_copy_roi_capped,50,50,1813,0.07,0.12,1.52,103284.02,0.42
11,avg_copy_roi_capped,100,100,2695,0.07,0.10,1.12,143225.93,0.42
0,prob_edge_shrunk,50,50,4204,0.01,0.12,0.93,115130.45,0.47
22,roi_sharpe,200,200,3250,0.01,0.07,0.85,101101.91,0.70
12,avg_copy_roi_capped,200,200,5272,0.06,0.11,0.82,248959.70,0.48
1,prob_edge_shrunk,100,100,5418,0.03,0.11,0.78,201409.43,0.51
13,avg_copy_roi_capped,300,300,7046,0.06,0.07,0.76,384078.84,0.50
23,roi_sharpe,300,300,4467,0.01,0.07,0.70,117444.18,0.70
6,weighted_prob_edge_shrunk,100,100,2399,0.06,0.09,0.68,256835.63,0.47
21,roi_sharpe,100,100,2473,0.01,0.03,0.66,73063.49,0.78


In [6]:
# pick best metric / top-N
best_row = cohort_sweep.sort_values(
    ["train_b_avg_copy_roi_capped", "train_b_weighted_prob_edge", "train_b_open_buy_trades"],
    ascending=[False, False, False],
).iloc[0]
BEST_SELECTION_METRIC = best_row["metric"]
BEST_TOP_N            = int(best_row["top_n"])
{"best_metric": BEST_SELECTION_METRIC, "best_top_n": BEST_TOP_N}


{'best_metric': 'avg_copy_roi_capped', 'best_top_n': 50}

## Select wallets (skill path) + build cohorts

In [7]:
selected_wallets = select_wallets(full_train_metrics, BEST_SELECTION_METRIC, BEST_TOP_N)
selected_wallets.to_parquet(WORKSPACE_DIR / "selected_wallets_v2.parquet", index=False)
selected_wallets[[
    "wallet", "open_buy_trades", "distinct_markets",
    "recent_open_buy_trades", BEST_SELECTION_METRIC, "wallet_quality"
]].head(15)


,wallet,open_buy_trades,distinct_markets,recent_open_buy_trades,avg_copy_roi_capped,wallet_quality
0,0x4d7fad0c5944fc24d4a67110f8e31abd5f559485,67,67,67,3.63,1.00
1,0xf94fe1799944b93b3b831b6f9f4f4aee1cdd78d7,27,27,27,2.73,0.98
2,0xbb0bd109b9f0c2a59b8819c466f064cf65ab3790,456,435,456,2.71,0.96
3,0x54a2c4cfc4332d831acc3f5a860d6540982c1d43,275,264,275,2.47,0.94
4,0x5ad5c4608c4661361b91c92e1091d2c5b43c37b9,464,451,464,2.41,0.92
5,0x1838cca016850ac7185a9b149fe7d0bd2d6629b4,851,221,851,2.32,0.90
6,0x82307f44c9405e73dc1cff466073dcc505535121,409,386,409,2.28,0.88
7,0x91b859901e1292380ef5da3c4f81b29738e551cf,32,31,32,2.18,0.86
8,0x456d7e1295b785ba37725a0db29436b056b11fe1,22,22,22,2.11,0.84
9,0x4266e8adab19c1c989547d7a4b4565a0336ba06d,38,34,38,2.06,0.82


## Build wallet profiles and signal events

In [8]:
from polymarket_analysis.signal.builder import (
    build_wallet_profiles,
    build_signal_events,
)

selected_wallet_profiles = build_wallet_profiles(
    dataset, selected_wallets, period="full_train",
    end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE
)
selected_wallet_profiles.to_parquet(
    WORKSPACE_DIR / "selected_wallet_profiles_v2.parquet", index=False
)

# Force-regenerate signal caches: the schema now includes all event types
# (open_buy, add_buy, close_sell, reduce_sell) plus copy_price/copy_market_key/
# copy_token_winner columns. Delete stale parquets so they are always rebuilt.
CALIBRATION_SIGNALS_PATH.unlink(missing_ok=True)
TEST_SIGNALS_PATH.unlink(missing_ok=True)

_, train_b_signals = build_signal_events(
    dataset, selected_wallet_profiles, period="train_b",
    end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
    price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
    tx_hash_column=TRIGGER_TX_HASH_COLUMN,
)
train_b_signals.to_parquet(CALIBRATION_SIGNALS_PATH, index=False)

_, test_signals = build_signal_events(
    dataset, selected_wallet_profiles, period="test",
    end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
    price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
    tx_hash_column=TRIGGER_TX_HASH_COLUMN,
)
test_signals.to_parquet(TEST_SIGNALS_PATH, index=False)

print(f"train_b signals: {len(train_b_signals):,}  test signals: {len(test_signals):,}")
print(f"event types: {train_b_signals['event_type'].value_counts().to_dict()}")

train_b signals: 4,789  test signals: 3,943
event types: {'add_buy': 2032, 'reduce_sell': 1939, 'open_buy': 664, 'close_sell': 148, 'other': 6}


## Calibrate signal scoring on train-B

In [9]:
from polymarket_analysis.signal.scorer import (
    build_calibration_tables,
    apply_signal_score,
    select_signal_threshold,
)

# Calibration tables are built from open_buy events only — the scoring model
# is designed for open-buy conviction features. Non-open-buy rows get neutral
# defaults and will not be used to calibrate the scorer.
train_b_open_buys = train_b_signals[train_b_signals["event_type"] == "open_buy"].copy()

price_table, consensus_table, global_edge = build_calibration_tables(train_b_open_buys)
train_b_scored = apply_signal_score(train_b_open_buys, price_table, consensus_table)
SIGNAL_THRESHOLD = select_signal_threshold(train_b_scored)
print(f"Global edge: {global_edge:.4f}")
print(f"Selected signal threshold: {SIGNAL_THRESHOLD:.2f}")

# score distribution
score_deciles = (
    train_b_scored.assign(
        score_decile=lambda df: pd.qcut(df["signal_score"], q=10, labels=False, duplicates="drop")
    )
    .groupby("score_decile")
    .agg(
        count=("signal_score", "size"),
        avg_signal_score=("signal_score", "mean"),
        avg_copy_roi_capped=("copy_roi_capped", "mean"),
    )
    .reset_index()
)
score_deciles

Global edge: 0.2205
Selected signal threshold: 0.65


,score_decile,count,avg_signal_score,avg_copy_roi_capped
0,0,67,0.36,1.18
1,1,68,0.46,1.39
2,2,64,0.53,0.69
3,3,67,0.61,1.36
4,4,66,0.69,2.33
5,5,66,0.74,3.51
6,6,67,0.77,2.32
7,7,67,0.82,2.19
8,8,88,0.86,0.97
9,9,44,0.90,1.54


## Build wallet cohorts (skill path)

In [10]:
from polymarket_analysis.wallet_selection.selector import build_wallet_cohorts

wallet_cohorts = build_wallet_cohorts(
    full_train_metrics, train_b_open_buys, selected_wallets,
    top_n=BEST_TOP_N,
)
{name: len(df) for name, df in wallet_cohorts.items()}


{'quality_core': 50, 'early_entry': 13, 'smooth_pnl': 50}

## Volatility-based wallet selection (second method)

Loads the full training set, applies the volatility filter, and stores the
result as an additional `WalletSelectionStrategy`.  The volatility wallet set
is added to `wallet_cohorts` so the strategy factory handles it uniformly.

In [11]:
dft = pd.read_parquet(TRADES_DIR / "0.parquet")


In [12]:
from polymarket_analysis.wallet_selection.volatility import (
    compute_wallet_metrics,
    filter_wallets_by_volatility,
)

# Load full training trades for the volatility path
trade_files = sorted(TRADES_DIR.glob("*.parquet"))
df_train_vol = pd.concat([pd.read_parquet(f) for f in trade_files], ignore_index=True)
df_train_vol["dt"] = pd.to_datetime(df_train_vol["dt"], utc=True)

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_train_vol.columns and "quantity" not in df_train_vol.columns:
    df_train_vol = df_train_vol.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_train_vol["usdc_amount"]      = df_train_vol["usdc_amount"].astype(float)
df_train_vol["final_value_usdc"] = df_train_vol["final_value_usdc"].astype(float)
df_train_vol["quantity"]         = df_train_vol["quantity"].astype(float)

# PnL and notional per fill

df_train_vol["copyable_pnl"] = (
        df_train_vol["copyable_qty"] / df_train_vol["quantity"]
        * (df_train_vol["final_price"] - df_train_vol["price"])
        * np.where(df_train_vol["side"] == "BUY", 1, -1)
    )

df_train_vol["pnl"] = np.where(
    df_train_vol["side"] == "BUY",
    df_train_vol["final_value_usdc"] - df_train_vol["usdc_amount"],
    df_train_vol["usdc_amount"] - df_train_vol["final_value_usdc"],
)
df_train_vol["notional"] = np.where(
    df_train_vol["side"] == "BUY",
    df_train_vol["usdc_amount"],
    df_train_vol["quantity"] * (1 - df_train_vol["price"].astype(float)),
)
df_slice = df_train_vol[df_train_vol["is_train"]].copy()

wallet_vol_train, _ = compute_wallet_metrics(df_slice)
# del df_train_vol, df_slice

In [49]:
wallet_vol_train['copyable_pnl_factor'] = np.clip(wallet_vol_train['copyable_pnl'] / wallet_vol_train['total_pnl'], 0, 1.0)
wallet_vol_train['copyable_roi'] = wallet_vol_train['average_roi'] * wallet_vol_train['copyable_pnl_factor']
wallet_vol_train.sort_values('copyable_roi', ascending=False, inplace=True)
wallet_vol_train.reset_index(drop=True, inplace=True)
wallet_vol_train = wallet_vol_train[wallet_vol_train['copyable_roi'] > 0.05]
wallet_vol_train = wallet_vol_train[wallet_vol_train['top_market_pnl_pct'] < 0.3]

In [79]:
wallet_cohorts = {}

In [80]:
print(len(wallet_vol_train))
wallet_vol_train.head()


407


,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top_market_pnl_pct,median_roi,average_roi,return,copyable_pnl_factor,copyable_roi
0,0x82307f44c9405e73dc1cff466073dcc505535121,9.46,488.00,386.00,40628.80,23369.47,10455.92,0.54,0.18,-1.00,4.93,0.58,0.45,2.21
1,0xede186f37ead91aa618cf8101db2bfe6b6c57dd1,5.59,49.00,33.00,9804.32,5952.92,5272.49,0.81,0.27,-1.00,1.30,0.61,0.89,1.15
2,0xfe0ad5ed4c912a46a4e41198e673bd371c384a63,24.85,138.00,40.00,96684.29,56567.43,22667.54,0.99,0.28,-1.00,2.64,0.59,0.40,1.06
3,0xde5830e0b75b31c466fc64aa2313a1e07e3f8610,2.12,13.00,11.00,473.49,791.49,417.97,0.76,0.24,1.78,1.66,1.67,0.53,0.88
4,0x8d5c3994aab4d5a04418d0f10394f3b398bab756,2.74,9.00,9.00,817.78,777.86,462.64,0.99,0.25,1.00,1.45,0.95,0.59,0.86


In [81]:
# filtered_wallets_vol = filter_wallets_by_volatility(
#     wallet_vol_train,
#     min_buckets=20,
#     max_top5_pnl_pct=.6,
#     max_top_market_pnl_pct=0.5,
# )

filtered_wallets_vol = wallet_vol_train.copy()

# filtered_wallets_vol = (
#     filtered_wallets_vol[
#         (filtered_wallets_vol['average_roi'] >= 0.04)
#         & (filtered_wallets_vol['copyable_pnl'] / filtered_wallets_vol['total_pnl'] >= 0.5)
# ].sort_values("pnl_volatility", ascending=True).head(100)
# )

print(f"Volatility-filtered wallets: {len(filtered_wallets_vol)}")

# Build wallet_quality based on total_pnl rank (higher = better)
filtered_wallets_vol = filtered_wallets_vol.copy().reset_index(drop=True)
filtered_wallets_vol["wallet_quality"] = filtered_wallets_vol["total_pnl"].rank(
    method="first", pct=True
)

# Add as additional cohort (uses only train-b signals for trigger calibration)
# We intersect with wallets that have signals to ensure coverage
vol_wallets_with_signals = set(train_b_signals["wallet"]).intersection(
    set(filtered_wallets_vol["wallet"])
)
wallet_cohorts["volatility"] = filtered_wallets_vol.copy().reset_index(drop=True)

# [
#     filtered_wallets_vol["wallet"].isin(vol_wallets_with_signals)
# ][["wallet", "wallet_quality"]].copy().reset_index(drop=True)

print(f"Volatility cohort: {len(wallet_cohorts['volatility'])}")

Volatility-filtered wallets: 407
Volatility cohort: 407


In [208]:
wallet_cohorts["top_pnl"] = wallet_vol_train.sort_values("total_pnl", ascending=False).head(100).copy().reset_index(drop=True)
wallet_cohorts["top_copyable_pnl"] = wallet_vol_train.sort_values("copyable_pnl", ascending=False).head(100).copy().reset_index(drop=True)
wallet_cohorts["top_copyable_roi"] = wallet_vol_train.sort_values("copyable_roi", ascending=False).head(100).copy().reset_index(drop=True)

wallet_cohorts['multi_filter'] = wallet_vol_train[
    (wallet_vol_train['copyable_roi'] > 0.05)
    &(wallet_vol_train['num_buckets'] >= 50)
    &(wallet_vol_train['copyable_pnl_factor'] >=0.4)
    & (wallet_vol_train['median_roi'] >= 0.05)
    & (wallet_vol_train['top_market_pnl_pct'] < 0.3)
    & (wallet_vol_train['top5_pnl_pct'] < 0.8)
    & (wallet_vol_train['total_pnl'] < 10_000)
].sort_values("copyable_roi", ascending=False).head(100).copy().reset_index(drop=True)

In [209]:
print(len(wallet_cohorts['multi_filter']))
pd.set_option('display.max_rows', 100)
wallet_cohorts["multi_filter"].sort_values("median_roi", ascending=True).head(100)

25


,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top_market_pnl_pct,median_roi,average_roi,return,copyable_pnl_factor,copyable_roi
9,0x7656ed7f597a0a61cd307591db198a42b2a7194b,5.57,215.00,68.00,32651.20,6087.97,3294.83,0.72,0.26,0.08,0.31,0.19,0.54,0.17
20,0x47c6e09427e5581445323964afa8432803e82fe4,2.53,253.00,136.00,22419.75,3255.73,1701.15,0.50,0.18,0.12,0.15,0.15,0.52,0.08
16,0x480e1e120dba94077cb8571fa72f6bc58c7971e9,4.58,298.00,145.00,57729.14,3685.75,2833.11,0.71,0.15,0.13,0.12,0.06,0.77,0.09
14,0x12e64c5230a1ba133d5b6406d1dd83c35739ea33,3.09,84.00,35.00,11481.20,1639.76,1172.60,0.78,0.27,0.16,0.16,0.14,0.72,0.11
18,0x54fde20fa60e932fc480071e05695b42caa94ac3,3.68,304.00,88.00,35840.35,5758.24,3746.00,0.52,0.13,0.25,0.13,0.16,0.65,0.08
21,0x5124088a00cde78aab67b8c3e47d2b98a8fa91bb,7.27,699.00,444.00,67245.83,8028.59,4222.16,0.76,0.17,0.25,0.12,0.12,0.53,0.07
11,0xaf3c7d23f497ea9aa23c522a25fa291274b4f5ff,1.53,80.00,32.00,10738.56,2958.11,1707.76,0.34,0.11,0.28,0.26,0.28,0.58,0.15
24,0x522f77e63f087115ee51c7cfda1d34c0b46e1415,0.83,377.00,275.00,9908.09,1144.01,583.34,0.31,0.09,0.28,0.10,0.12,0.51,0.05
7,0x0269f11e5f0956d89417f9b508a35b2a16dc783b,2.99,56.00,48.00,6017.29,1184.57,764.01,0.75,0.20,0.29,0.27,0.20,0.64,0.18
23,0x8b552440eaefeef7d4818e6d74cf5c391a76d74b,1.70,486.00,284.00,22337.21,2319.07,1147.14,0.44,0.17,0.29,0.11,0.10,0.49,0.06


## Build and save strategy registry

All cohorts × all trigger variants → persisted `WalletSelectionStrategy` files.
The backtest loads these directly.

In [210]:
from polymarket_analysis.wallet_selection.selector import build_strategies_from_sweep
from polymarket_analysis.strategy.registry import save_strategy

all_strategies = build_strategies_from_sweep(
    wallet_cohorts=wallet_cohorts,
    signal_threshold=SIGNAL_THRESHOLD,
    selection_metric=BEST_SELECTION_METRIC,
    top_n=BEST_TOP_N,
    sweep_df=cohort_sweep,
    extra_metadata={
        "end_date_train": str(END_DATE_TRAIN),
        "train_a_end_date": str(TRAIN_A_END_DATE),
    },
)

for strategy in all_strategies:
    parquet_path, json_path = save_strategy(strategy, WORKSPACE_DIR)
    print(f"Saved [{strategy.strategy_id}]  wallets={len(strategy.wallets):3d}  trigger={strategy.trigger.fn_ref.split('.')[-1]}")

print(f"\nTotal strategies saved: {len(all_strategies)}")


Saved [volatility__score_threshold]  wallets=407  trigger=score_threshold
Saved [volatility__all_open_buys]  wallets=407  trigger=all_open_buys
Saved [volatility__copy_triggers]  wallets=407  trigger=copy_triggers
Saved [top_pnl__score_threshold]  wallets=100  trigger=score_threshold
Saved [top_pnl__all_open_buys]  wallets=100  trigger=all_open_buys
Saved [top_pnl__copy_triggers]  wallets=100  trigger=copy_triggers
Saved [top_copyable_pnl__score_threshold]  wallets=100  trigger=score_threshold
Saved [top_copyable_pnl__all_open_buys]  wallets=100  trigger=all_open_buys
Saved [top_copyable_pnl__copy_triggers]  wallets=100  trigger=copy_triggers
Saved [top_copyable_roi__score_threshold]  wallets=100  trigger=score_threshold
Saved [top_copyable_roi__all_open_buys]  wallets=100  trigger=all_open_buys
Saved [top_copyable_roi__copy_triggers]  wallets=100  trigger=copy_triggers
Saved [multi_filter__score_threshold]  wallets= 25  trigger=score_threshold
Saved [multi_filter__all_open_buys]  wall

## Strategy registry summary

In [211]:
from polymarket_analysis.strategy.registry import load_all_strategies

registry = load_all_strategies(WORKSPACE_DIR)
summary_rows = []
for sid, s in registry.items():
    summary_rows.append({
        "strategy_id": s.strategy_id,
        "name": s.name,
        "selection_method": s.selection_method,
        "num_wallets": len(s.wallets),
        "trigger_fn": s.trigger.fn_ref.split(".")[-1],
        "threshold": s.trigger.params.get("threshold"),
        "dynamic_sizing": s.trigger.params.get("dynamic_sizing"),
    })

pd.DataFrame(summary_rows)


,strategy_id,name,selection_method,num_wallets,trigger_fn,threshold,dynamic_sizing
0,early_entry__all_open_buys,early_entry | all open-buys (fixed stake),cohort_early_entry,13,all_open_buys,NaN,False
1,early_entry__copy_triggers,early_entry | copy all triggers (tight slippage),cohort_early_entry,13,copy_triggers,NaN,False
2,early_entry__score_threshold,early_entry | score >= 0.65 (Kelly),cohort_early_entry,13,score_threshold,0.65,True
3,last_ten__all_open_buys,last_ten | all open-buys (fixed stake),last_ten,10,all_open_buys,NaN,False
4,last_ten__copy_triggers,last_ten | copy all triggers (tight slippage),last_ten,10,copy_triggers,NaN,False
5,last_ten__score_threshold,last_ten | score >= 0.65 (Kelly),last_ten,10,score_threshold,0.65,True
6,multi_filter__all_open_buys,multi_filter | all open-buys (fixed stake),multi_filter,25,all_open_buys,NaN,False
7,multi_filter__copy_triggers,multi_filter | copy all triggers (tight slippage),multi_filter,25,copy_triggers,NaN,False
8,multi_filter__score_threshold,multi_filter | score >= 0.65 (Kelly),multi_filter,25,score_threshold,0.65,True
9,quality_core__all_open_buys,quality_core | all open-buys (fixed stake),skill_sweep,50,all_open_buys,NaN,False


## Wallet PnL over time

Three independent plots:
- **Training plot** — cohort aggregate cumulative PnL over the training period only
- **Test plot** — cohort aggregate cumulative PnL over the test period only (starts from 0)
- **Individual plot** — per-wallet cumulative PnL spanning train + test, with a train/test split
  vline and wallet address labels; test portion resets to 0 at the split boundary

Set `PLOT_WALLET_PNL = False` to skip (e.g. when running headlessly).


In [212]:
PLOT_WALLET_PNL  = True
TOP_N_INDIVIDUAL = 10   # wallets shown in panel 1 per cohort


In [213]:
_trade_files = sorted(TRADES_DIR.glob("*.parquet"))
df_fills = pd.concat([pd.read_parquet(f) for f in _trade_files], ignore_index=True)
df_fills["dt"]               = pd.to_datetime(df_fills["dt"], utc=True)

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_fills.columns and "quantity" not in df_fills.columns:
    df_fills = df_fills.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_fills["usdc_amount"]      = df_fills["usdc_amount"].astype(float)
df_fills["final_value_usdc"] = df_fills["final_value_usdc"].astype(float)
df_fills["quantity"]         = df_fills["quantity"].astype(float)

In [214]:
len(wallet_cohorts['volatility'])

407

In [215]:
wallet_cohorts['volatility']['wallet'].to_list()

['0x82307f44c9405e73dc1cff466073dcc505535121',
 '0xede186f37ead91aa618cf8101db2bfe6b6c57dd1',
 '0xfe0ad5ed4c912a46a4e41198e673bd371c384a63',
 '0xde5830e0b75b31c466fc64aa2313a1e07e3f8610',
 '0x8d5c3994aab4d5a04418d0f10394f3b398bab756',
 '0x9bc8e9f0c3ada70da1a82f3051448b6cc4783361',
 '0x071682d3d8ea0e9e7f69e370b47830109c0b5ff1',
 '0x6ef453ffe9139352188648263d1e644aede783f4',
 '0x7eb134f39c4809a15cf8a41218a2a64c8c45fdb2',
 '0x126073a6eca0937c642a6f69f0c8e98030efb85a',
 '0x4427b362e3c00a3dc41d0d0eea1ec86bd0880e7f',
 '0xb89190c659af6087669fce8a4c9dc0f2102e969b',
 '0xb22d3e8d06bed1d38f5f0336d43c0b2ab5d5b530',
 '0xbdc2b2d3eaf1cc272267127b74d841d6cfe3e7c9',
 '0xe0c589813f2bf5315171a035604f64178ca77aa9',
 '0x5f59a5196372d454d62d51ba5f1a83b2cd88152c',
 '0x4162cf0487763d777bfdfbe6a54c5fa2bccf5a99',
 '0x963e75d62f11939a96d0fcc1a67bafac29004383',
 '0xf8fc0d2cc44fcdc74e1bab5590d49bfe812704aa',
 '0x08a807c407cad0a33646243bb598889007844c60',
 '0x1190b18ddbf41ee89798077c42fce20f261f936a',
 '0x78c433467

In [216]:
vol_fills = df_fills[
    (df_fills["wallet"].isin(wallet_cohorts["volatility"]["wallet"]))
    & (df_fills["is_train"] == False)
].sort_values("dt")
len(vol_fills)

110298

In [217]:
# last_ten = ['0x6fbbd19e716f7454bdf483c41e6fa276a9e139da',
#  '0x47c6e09427e5581445323964afa8432803e82fe4',
#  '0x3f2c9b3d2d7a09143f810940b4da51ac90b2f0cf',
#  '0x8b0244a0ded9943470a6bdad9050dce24bb1f3d9',
#  '0x0c4f5b1295f39cf505679209a22adbafe61c0f33',
#  '0xaf3c7d23f497ea9aa23c522a25fa291274b4f5ff',
#  '0xbff9efe3a976c115e3e639b4c6b9c7168479009d',
#  '0x236599e3745dbea9d285d7ac967846f476d8aaf4',
#  '0xd6f44883f664d7dc963d8b89c5a0689fdd330566',
#  '0xf201a19b43471261a3c1ba9247335d55270e527e']
# top_ten =['0xabde435151e5cf858071cec380303acb610fd41f',
#  '0xec593b3c1a9a0ed15e597dc4ea09cafbffe3bb35',
#  '0x37bdf9681cecfebe1f5709d2aed823e0063c2a26',
#  '0x702226b14b65b1c8d05ff51759e095f09932f64e',
#  '0xcf0aca0d7a395202aec661c3666be9cc098e320a',
#  '0xdaa6a2cd4ba545befb3dbdc25d2b444c46873e62',
#  '0x4a4b5d59f1979fdf45fa15c17e2ba414f2ebbbca',
#  '0x2687cb380661a0041bf4c4cf3945a034b79e1363',
#  '0xc5b5bbd42624a8f0c8dfa90221913007d8c77e80',
#  '0x3335de45972fd32e5d8a3b689e772147f2693656']
# vol = wallet_cohorts['volatility']
# last_ten_vol = vol[vol['wallet'].isin(last_ten)].copy().reset_index(drop=True)
# top_ten_vol = vol[vol['wallet'].isin(top_ten)].copy().reset_index(drop=True)
# wallet_cohorts['last_ten'] = last_ten_vol
# wallet_cohorts['top_ten'] = top_ten_vol

In [218]:
from polymarket_analysis.visualization.wallet_plots import (
    plot_wallet_selection_pnl,
    plot_wallet_individual_pnl,
)

if PLOT_WALLET_PNL:
    _trade_files = sorted(TRADES_DIR.glob("*.parquet"))
    df_fills = pd.concat([pd.read_parquet(f) for f in _trade_files], ignore_index=True)
    df_fills["dt"]               = pd.to_datetime(df_fills["dt"], utc=True)

    # Normalise grouped schema → canonical fill-level names
    if "total_quantity" in df_fills.columns and "quantity" not in df_fills.columns:
        df_fills = df_fills.rename(columns={
            "total_quantity": "quantity",
            "avg_price": "price",
            "trade_value_usdc": "usdc_amount",
        })

    df_fills["usdc_amount"]      = df_fills["usdc_amount"].astype(float)
    df_fills["final_value_usdc"] = df_fills["final_value_usdc"].astype(float)
    df_fills["quantity"]         = df_fills["quantity"].astype(float)

    split_dt = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

    # Cohort aggregate PnL — training period
    fig_train = plot_wallet_selection_pnl(
        df_fills,
        wallet_cohorts,
        split_date=split_dt,
        period="train",
        title="Wallet selection cohorts — cohort cumulative PnL (training period)",
    )
    fig_train.show(renderer="browser")

    # Cohort aggregate PnL — test period (starts from 0)
    fig_test = plot_wallet_selection_pnl(
        df_fills,
        wallet_cohorts,
        split_date=split_dt,
        period="test",
        title="Wallet selection cohorts — cohort cumulative PnL (test period)",
    )
    fig_test.show(renderer="browser")

    # # Individual wallet lines (train + test) with split vline and labels
    # fig_ind = plot_wallet_individual_pnl(
    #     df_fills,
    #     wallet_cohorts,
    #     split_date=split_dt,
    #     top_n_individual=TOP_N_INDIVIDUAL,
    #     title="Individual wallet cumulative PnL (train + test, resets at split)",
    # )
    # fig_ind.show(renderer="browser")

    del df_fills
else:
    print("PLOT_WALLET_PNL=False — skipping plots")
